In [2]:
import pandas as pd
import numpy as np

# This is the file you are reading
file_path = 'output_Pune_builderfloor.csv'

In [4]:
df = pd.read_csv(file_path)
print("--- Initial Data Info ---")
df.info()

--- Initial Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   location               123 non-null    object 
 1   area                   123 non-null    int64  
 2   price                  123 non-null    int64  
 3   price_currency         123 non-null    object 
 4   status                 41 non-null     float64
 5   new/resale             123 non-null    int64  
 6   price_negotiable       123 non-null    int64  
 7   description            118 non-null    object 
 8   security_deposit       123 non-null    int64  
 9   facing                 76 non-null     object 
 10  furnished              123 non-null    int64  
 11  age of property        123 non-null    int64  
 12  Lift(s)                123 non-null    int64  
 13  Full Power Backup      123 non-null    int64  
 14  24 X 7 Security        123 non-n

### droping col with > 70 % null

In [8]:
missing_percentage = df.isnull().mean()
cols_to_drop_auto = missing_percentage[missing_percentage > 0.7].index

print(f"\n--- Dropping {len(cols_to_drop_auto)} columns with >70% nulls ---")
print(list(cols_to_drop_auto))


--- Dropping 12 columns with >70% nulls ---
['project_score', 'Vaastu Compliant', 'Indoor Games', 'Multipurpose Room', 'Staff Quarter', 'Cafeteria', 'School', 'Maintenance Staff', 'Shopping Mall', 'Golf Course', 'ATM', 'Hospital']


In [9]:
cols_to_drop_manual = [
    'description',       # Unstructured text
    'price_currency',    # Only one value (INR)
    'status'             # Too many nulls and complex
]

# Combine the lists and drop them all
all_cols_to_drop = cols_to_drop_auto.union(cols_to_drop_manual)
df_cleaned = df.drop(columns=all_cols_to_drop)

print(f"\n--- Data shape after dropping: {df_cleaned.shape} ---")
print("Remaining columns:")
print(list(df_cleaned.columns))


--- Data shape after dropping: (123, 24) ---
Remaining columns:
['location', 'area', 'price', 'new/resale', 'price_negotiable', 'security_deposit', 'facing', 'furnished', 'age of property', 'Lift(s)', 'Full Power Backup', '24 X 7 Security', "Children's play area", 'Club House', 'Gymnasium', 'Swimming Pool', 'Sports Facility', 'Jogging Track', 'Landscaped Gardens', 'locality_score', 'builder_experience', 'Rain Water Harvesting', 'Intercom', 'Car Parking']


In [11]:
df_cleaned['facing'] = df_cleaned['facing'].fillna('Unknown')
print("\nFilled 'facing' nulls with 'Unknown'.")


Filled 'facing' nulls with 'Unknown'.


In [13]:
num_cols_with_na = ['locality_score', 'builder_experience']

for col in num_cols_with_na:
    if col in df_cleaned.columns:
        median_val = df_cleaned[col].median()
        df_cleaned[col] = df_cleaned[col].fillna(median_val)
        print(f"Filled '{col}' nulls with median value: {median_val}")

Filled 'locality_score' nulls with median value: 8.8
Filled 'builder_experience' nulls with median value: 89.5


In [16]:
remaining_na_cols = df_cleaned.columns[df_cleaned.isnull().any()]
print(f"\n--- Filling remaining amenity nulls with 0 ---")
print(list(remaining_na_cols))

df_cleaned = df_cleaned.fillna(0)


--- Filling remaining amenity nulls with 0 ---
[]


In [21]:
print(f"\nTotal nulls remaining: {df_cleaned.isnull().sum().sum()}")


Total nulls remaining: 0


### feature engineering and creating USPs and calculating key values

In [19]:
df_cleaned['price_per_sqft'] = df_cleaned['price'] / df_cleaned['area']
print("\nCreated 'price_per_sqft' feature.")


Created 'price_per_sqft' feature.


In [22]:
# List of all possible amenity columns from the original file
amenity_cols = [
    'Lift(s)', 'Full Power Backup', '24 X 7 Security', "Children's play area",
    'Club House', 'Gymnasium', 'Swimming Pool', 'Sports Facility', 'Jogging Track',
    'Landscaped Gardens', 'Rain Water Harvesting', 'Intercom', 'Car Parking'
]

# Filter list to only columns that *actually* exist in our cleaned frame
amenity_cols_present = [col for col in amenity_cols if col in df_cleaned.columns]

df_cleaned['amenities_score'] = df_cleaned[amenity_cols_present].sum(axis=1)
print("Created 'amenities_score' feature.")

Created 'amenities_score' feature.


In [23]:
df_processed = pd.get_dummies(df_cleaned, columns=['location', 'facing'], drop_first=True)

print(f"\n--- Data shape after encoding: {df_processed.shape} ---")
print("Ready for modeling.")


--- Data shape after encoding: (123, 78) ---
Ready for modeling.


In [28]:
deliverable_file = 'processed_data.csv'
df_processed.to_csv(deliverable_file, index=False)

print(f"Cleaned data saved to: {deliverable_file}")


Cleaned data saved to: processed_data.csv


In [29]:
df2 = pd.read_csv('processed_data.csv')
df2.head()

,area,price,new/resale,price_negotiable,security_deposit,furnished,age of property,Lift(s),Full Power Backup,24 X 7 Security,...,location_Wadgaon Sheri,location_Wagholi,location_Wakad,location_Warje,facing_east,facing_north,facing_northeast,facing_south,facing_southeast,facing_west
0,832,6500000,0,0,1,0,0,0,0,0,...,False,False,False,True,False,False,False,False,False,False
1,800,6000000,0,0,1,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,849,3399396,0,0,1,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
3,1550,20498750,0,0,1,0,0,1,0,0,...,False,False,False,False,False,True,False,False,False,False
4,850,6400000,0,0,1,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
